In [61]:
import pandas as pd
import numpy as np
from scipy.io import loadmat
from pathlib import Path
import os
import mne

In [ ]:
data_folder = Path('../Data/')
raw_data_folder = Path('../Data/raw_data/')

# Reading Data

In [35]:
scales = pd.read_excel(data_folder / 'scales.xls', header=[0, 1])

In [36]:
scales.info()

<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 10 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   (Subject No., Unnamed: 0_level_1)  40 non-null     int64
 1   (Trial_1, Maths)                   40 non-null     int64
 2   (Trial_1, Symmetry)                40 non-null     int64
 3   (Trial_1, Stroop)                  40 non-null     int64
 4   (Trial_2, Maths)                   40 non-null     int64
 5   (Trial_2, Symmetry)                40 non-null     int64
 6   (Trial_2, Stroop)                  40 non-null     int64
 7   (Trial_3, Maths)                   40 non-null     int64
 8   (Trial_3, Symmetry)                40 non-null     int64
 9   (Trial_3, Stroop)                  40 non-null     int64
dtypes: int64(10)
memory usage: 3.3 KB


In [37]:
scales.head()

Subject No. Trial_1                 Trial_2                 Trial_3  \
  Unnamed: 0_level_1   Maths Symmetry Stroop   Maths Symmetry Stroop   Maths   
0                  1       6        3      3       7        5      2       4   
1                  2       3        4      5       3        4      4       7   
2                  3       5        3      4       3        5      5       8   
3                  4       5        3      4       3        5      2       7   
4                  5       6        6      6       5        3      2       5   

                   
  Symmetry Stroop  
0        7      4  
1        5      3  
2        7      5  
3        5      5  
4        7      3

` It is complex so I make it a simple lookup table`

In [38]:
scales.columns = [
    f'{trial}_{task}'
    if trial != 'Subject No.'
    else "Subject_No"
    for trial, task in scales.columns
]
scales.set_index('Subject_No', inplace=True)

In [39]:
scales.head()

,Trial_1_Maths,Trial_1_Symmetry,Trial_1_Stroop,Trial_2_Maths,Trial_2_Symmetry,Trial_2_Stroop,Trial_3_Maths,Trial_3_Symmetry,Trial_3_Stroop
Subject_No,,,,,,,,,
1,6,3,3,7,5,2,4,7,4
2,3,4,5,3,4,4,7,5,3
3,5,3,4,3,5,5,8,7,5
4,5,3,4,3,5,2,7,5,5
5,6,6,6,5,3,2,5,7,3


In [40]:
subject1 = scales.loc[1]
subject1

Trial_1_Maths       6
Trial_1_Symmetry    3
Trial_1_Stroop      3
Trial_2_Maths       7
Trial_2_Symmetry    5
Trial_2_Stroop      2
Trial_3_Maths       4
Trial_3_Symmetry    7
Trial_3_Stroop      4
Name: 1, dtype: int64

# Create A Master DataFrame

In [65]:
data = []
for file in raw_data_folder.glob("*.mat"):
    name = file.name
    if name.startswith('Mirror'):
        name = name.replace('_image', '')

    # print(name)
    kind = name.split('_')[0]
    subject = int(name.split('_')[2])
    trial = int(name.split('_')[3][5:6])

    sample = {
        "subject": subject,
        "trial": trial,
        "kind": kind,
        "file_name": file.name
    }

    data.append(sample)

df = pd.DataFrame(data)

In [66]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   subject    480 non-null    int64
 1   trial      480 non-null    int64
 2   kind       480 non-null    str  
 3   file_name  480 non-null    str  
dtypes: int64(2), str(2)
memory usage: 15.1 KB


In [67]:
df.head()

,subject,trial,kind,file_name
0,6,2,Arithmetic,Arithmetic_sub_6_trial2.mat
1,10,1,Relax,Relax_sub_10_trial1.mat
2,36,3,Relax,Relax_sub_36_trial3.mat
3,27,1,Relax,Relax_sub_27_trial1.mat
4,17,2,Stroop,Stroop_sub_17_trial2.mat


In [68]:
df.sort_values(by=['subject', 'trial'], inplace=True)
df.head(5)

,subject,trial,kind,file_name
149,1,1,Relax,Relax_sub_1_trial1.mat
161,1,1,Mirror,Mirror_image_sub_1_trial1.mat
214,1,1,Arithmetic,Arithmetic_sub_1_trial1.mat
329,1,1,Stroop,Stroop_sub_1_trial1.mat
127,1,2,Mirror,Mirror_image_sub_1_trial2.mat


In [69]:
df['kind'].value_counts()

kind
Relax         120
Mirror        120
Arithmetic    120
Stroop        120
Name: count, dtype: int64

In [70]:
def extract_stress_level(scores: pd.DataFrame, subject: int, trial: int, kind: str) -> int:
    subject = scores.loc[subject]
    mapping = {
        'Arithmetic': 'Maths',
        'Mirror': 'Symmetry',
        'Stroop': 'Stroop'
    }
    if kind not in mapping:
        return 0
    
    kind = mapping[kind]
    column_name = f'Trial_{trial}_{kind}'
    return subject[column_name]

In [71]:
df['stress_level'] = df.apply(
    lambda row: extract_stress_level(
        scales,
        row['subject'],
        row['trial'],
        row['kind']
    ),
    axis=1
)

In [72]:
df.head()

,subject,trial,kind,file_name,stress_level
149,1,1,Relax,Relax_sub_1_trial1.mat,0
161,1,1,Mirror,Mirror_image_sub_1_trial1.mat,3
214,1,1,Arithmetic,Arithmetic_sub_1_trial1.mat,6
329,1,1,Stroop,Stroop_sub_1_trial1.mat,3
127,1,2,Mirror,Mirror_image_sub_1_trial2.mat,5


In [73]:
df.to_csv(data_folder / 'preprocessed_data.csv', index=False)

In [74]:
df.info()

<class 'pandas.DataFrame'>
Index: 480 entries, 149 to 466
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   subject       480 non-null    int64
 1   trial         480 non-null    int64
 2   kind          480 non-null    str  
 3   file_name     480 non-null    str  
 4   stress_level  480 non-null    int64
dtypes: int64(3), str(2)
memory usage: 22.5 KB
